<a href="https://colab.research.google.com/github/Jahnavi-Rav/vLLM-Inference-Profiling/blob/main/vllm_inference_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1: Imports
from vllm import LLM, SamplingParams

# CELL 2: Load model WITH profiling enabled from the start
import os
os.makedirs("/tmp/vllm_traces", exist_ok=True)  # Create folder to save trace files

llm = LLM(
    model="TheBloke/Mistral-7B-Instruct-v0.2-AWQ",
    quantization="awq",
    dtype="float16",
    profiler_config={                          # ← ADD THIS
        "profiler": "torch",
        "torch_profiler_dir": "/tmp/vllm_traces"
    }
)

INFO 03-29 01:27:28 [utils.py:233] non-default args: {'dtype': 'float16', 'disable_log_stats': True, 'quantization': 'awq', 'profiler_config': ProfilerConfig(profiler='torch', torch_profiler_dir='/tmp/vllm_traces', torch_profiler_with_stack=False, torch_profiler_with_flops=False, torch_profiler_use_gzip=True, torch_profiler_dump_cuda_time_total=True, torch_profiler_record_shapes=False, torch_profiler_with_memory=False, ignore_frontend=False, delay_iterations=0, max_iterations=0, warmup_iterations=0, active_iterations=5, wait_iterations=0), 'model': 'TheBloke/Mistral-7B-Instruct-v0.2-AWQ'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 03-29 01:27:30 [model.py:533] Resolved architecture: MistralForCausalLM
INFO 03-29 01:27:30 [model.py:1582] Using max model len 32768
INFO 03-29 01:27:32 [awq_marlin.py:166] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 03-29 01:27:32 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-29 01:27:32 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 03-29 01:27:34 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-29 01:29:12 [llm.py:391] Supported tasks: ['generate']


In [2]:
# CELL 3: Run inference and measure stats
import time

prompts = ["Explain transformer attention mechanism in detail"] * 8
sampling_params = SamplingParams(temperature=0.7, max_tokens=200)

# --- BEFORE: Measure baseline (no profiling overhead) ---
print("🔥 Running inference...")
start_time = time.time()

llm.start_profile()
outputs = llm.generate(prompts, sampling_params)
llm.stop_profile()

end_time = time.time()
total_time = end_time - start_time

# --- Calculate Stats ---
total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
throughput = total_tokens / total_time
avg_tokens_per_request = total_tokens / len(prompts)
ttft_approx = total_time / len(prompts)  # rough estimate

print("\n" + "="*50)
print("📊 PROFILING RESULTS")
print("="*50)
print(f"✅ Total requests:        {len(prompts)}")
print(f"✅ Total tokens generated: {total_tokens}")
print(f"✅ Total time:            {total_time:.2f} seconds")
print(f"✅ Throughput:            {throughput:.1f} tokens/sec")
print(f"✅ Avg tokens/request:    {avg_tokens_per_request:.0f}")
print(f"✅ Approx latency/req:    {ttft_approx:.2f} seconds")
print(f"\n📁 Trace saved to: /tmp/vllm_traces/")
print("   → Download and drag into https://ui.perfetto.dev to visualize!")

🔥 Running inference...


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


📊 PROFILING RESULTS
✅ Total requests:        8
✅ Total tokens generated: 1600
✅ Total time:            79.30 seconds
✅ Throughput:            20.2 tokens/sec
✅ Avg tokens/request:    200
✅ Approx latency/req:    9.91 seconds

📁 Trace saved to: /tmp/vllm_traces/
   → Download and drag into https://ui.perfetto.dev to visualize!


In [4]:
# CELL 4 (REPLACEMENT): Test different batch sizes - real optimization on T4
import time

def benchmark_run(llm_instance, num_prompts, label):
    prompts = ["Explain transformer attention mechanism in detail"] * num_prompts
    sampling_params = SamplingParams(temperature=0.7, max_tokens=200)

    start = time.time()
    outputs = llm_instance.generate(prompts, sampling_params)
    elapsed = time.time() - start

    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_tokens / elapsed
    latency = elapsed / num_prompts

    print(f"\n{'='*45}")
    print(f"🔧 Config: {label}")
    print(f"   Requests:    {num_prompts}")
    print(f"   Throughput:  {throughput:.1f} tokens/sec")
    print(f"   Latency/req: {latency:.2f} sec")
    print(f"   Total time:  {elapsed:.2f} sec")
    return throughput, latency

# Test 1: Already done — 8 requests (your baseline)
print("✅ BASELINE already recorded:")
print("   8 requests | 20.2 tok/s | 9.91 sec/req")

# Test 2: Single request — shows per-request latency without batching
t2, l2 = benchmark_run(llm, 1, "Single Request (no batching)")

# Test 3: 4 requests — medium batch
t3, l3 = benchmark_run(llm, 4, "4 Requests (medium batch)")

# Final comparison table
print("\n" + "="*55)
print("📊 FINAL COMPARISON TABLE")
print("="*55)
print(f"{'Config':<25} {'Throughput':>12} {'Latency/req':>12}")
print("-"*55)
print(f"{'1 request':<25} {t2:>10.1f}   {l2:>10.2f}s")
print(f"{'4 requests':<25} {t3:>10.1f}   {l3:>10.2f}s")
print(f"{'8 requests':<25} {'20.2':>10}   {'9.91':>10}s")
print(f"\n💡 Insight: Batching improves throughput by {20.2/t2:.1f}x vs single request!")

✅ BASELINE already recorded:
   8 requests | 20.2 tok/s | 9.91 sec/req


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


🔧 Config: Single Request (no batching)
   Requests:    1
   Throughput:  3.1 tokens/sec
   Latency/req: 64.22 sec
   Total time:  64.22 sec


Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


🔧 Config: 4 Requests (medium batch)
   Requests:    4
   Throughput:  13.6 tokens/sec
   Latency/req: 14.68 sec
   Total time:  58.73 sec

📊 FINAL COMPARISON TABLE
Config                      Throughput  Latency/req
-------------------------------------------------------
1 request                        3.1        64.22s
4 requests                      13.6        14.68s
8 requests                      20.2         9.91s

💡 Insight: Batching improves throughput by 6.5x vs single request!
